# B.L.U.S.H: ACT on dataset v6 (three cameras), batch 64, constant learning rate, no dropout

**Prepared 2026-11-26. NOT RUN YET: it needs K7's go and the v6 dataset.** The v5b64 notebook
(`colab_train_act.ipynb`) stays as the record of that run.

What is the same as v5b64 (K7 2026-11-26): **batch 64**, **learning rate 3e-5**, AdamW, weight decay 1e-4, image
augmentation on, torchcodec decoding, full precision, 100k steps, a checkpoint every 5k.

What changed, each one a fix from the v5 post-mortem (vault note, sections 7 and 8):
- **Dropout off** (fix 2). v5 trained with 10 % dropout and its running answers came out about 3.5 % "too far
  out" against its training-mode answers. Cell 6 checks on the smoke checkpoint that the two answers are now the
  same ("gate b"); the laptop repeats it on real checkpoints.
- **A constant learning rate** (fix 6, step 3): no warmup and no cosine decay. This is ACT's own preset in lerobot
  (AdamW, no scheduler), so `--use_policy_training_preset` stays on and the rate is set on the policy:
  `--policy.optimizer_lr=3e-5` and `--policy.optimizer_lr_backbone=3e-5`. ⚠️ The backbone flag matters: with the
  preset ON the picture backbone has its own rate (default 1e-5); in v5b64 the preset was off and every weight ran
  at the 3e-5 peak (lerobot `optim/factory.py`, read 2026-11-26). Cell 6 checks that the logged rate does not move.
- **Pictures stored so that a test can reproduce them** (fix 1). The v6 dataset is recorded through
  `sim/picture_store.py` (H.264, every frame by itself, fixed quality step). Cell 4 runs the dataset's picture
  gate here: frames decoded on this machine must match the fingerprints the recorder wrote. The v5 fingerprints
  are gone from this notebook.
- **Three cameras** (two fixed + the claw). Nothing to set here: the policy takes its inputs from the dataset.
  ⚠️ Not measured: GPU memory and speed with a third 640 x 480 picture. Cell 6 prints both; if the A100 runs out
  of memory, tell K7 (do not lower the batch by yourself: batch 64 is the recipe).

**Dataset:** `DATASET` in cell 3 is a placeholder name until the v6 dataset exists (K7 2026-11-26: the 800 old
episodes with their pictures re-made for the new cameras + 600 new ones, about 670k frames; 100k steps of 64 is
then about 9.5 passes over every frame, v5b64 had 17, v5 about 2). One old episode (353) does not rebuild on
MuJoCo 4.0.1, so the re-made part is 799 episodes (`sim/README.md`, Dataset v6).

**Where things go**
- The dataset goes from the Hub straight to this machine's local disk (cell 4), never through Drive or the laptop.
- Every 5k steps a checkpoint goes to the private Hub repo `MODEL_REPO` (lerobot does this itself, and it is how the
  laptop's tests get the weights) and to Google Drive (`DRIVE_DIR`): the weights of every checkpoint, the full
  resumable state of 50k and of the newest one.
- The laptop pulls each checkpoint from the Hub and tests it in the simulator (`sim/scripts/colab_tests.sh`), with
  the dataset's own picture path (automatic), both guards outside the policy (the scripts' default since 2026-11-27) and the clean-cut measure next to plain success.

**Before you run:** Runtime -> Change runtime type -> **A100** (G4 if available). Secrets (key icon): `HF_TOKEN` with
write access, "Notebook access" on. Keep the tab open: a closed tab ends the session.

**After a disconnect:** run all again. Cell 7 finds the newest checkpoint on the Hub and resumes from it.


In [ ]:
# 1. The machine: GPU, CPU cores, RAM, local disk.
import os, shutil, subprocess, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)
CORES = os.cpu_count()
ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
print(f"python {sys.version.split()[0]} | {CORES} CPU cores | {ram:.0f} GB RAM | "
      f"local disk free {shutil.disk_usage('/content').free / 1e9:.0f} GB")
WORK = "/content"

In [ ]:
# 2. lerobot, pinned to the SAME commit as the laptop (0.6.2, 5aa74557), so the checkpoints load in the local tests.
#    Then a real computation on the GPU: a torch build without this GPU's architecture still says "cuda available".
LEROBOT = "lerobot[training] @ git+https://github.com/huggingface/lerobot@5aa74557"
%pip install -q "{LEROBOT}"
TRAIN = shutil.which("lerobot-train") or "lerobot-train"
CHECK = ("import torch, torchcodec, lerobot; x = torch.randn(512, 512, device='cuda'); "
         "y = (x @ x).to(torch.bfloat16).float().sum().item(); "
         "print('lerobot', lerobot.__version__, 'torch', torch.__version__, 'torchcodec', torchcodec.__version__, "
         "'gpu', torch.cuda.get_device_name(0), 'capability', torch.cuda.get_device_capability(0), 'matmul ok', y == y)")
r = subprocess.run([sys.executable, "-c", CHECK], capture_output=True, text=True)
print(r.stdout, r.stderr[-3000:])
assert r.returncode == 0 and "matmul ok True" in r.stdout, "torch can't compute on this GPU: read the lines above"

In [ ]:
# 3. Hugging Face login (token from Colab Secrets, never from the notebook) and the run's settings.
from google.colab import userdata
from huggingface_hub import whoami
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("logged in as", whoami()["name"])

RUN = "act_teacher_v6b64"
DATASET = "krish5831/blush_sim_teacher_v6"        # ⚠️ placeholder: set it to the v6 dataset's real name (private)
MODEL_REPO = "krish5831/blush_sim_act_v6b64"      # every checkpoint under checkpoints/<step>/ (private)
DRIVE_DIR = f"/content/drive/MyDrive/blush_training/{RUN}"
KEEP_STATE = {"050000"}                            # Drive keeps these steps' optimizer state too (K7's 50k stop)
STEPS, SAVE_FREQ = 100_000, 5_000
BATCH = 64                                         # K7 2026-11-26: batch 64 and 3e-5 again
LR = 3e-5                                          # constant: ACT's preset has no scheduler (post-mortem fix 6)
LR_BACKBONE = 3e-5                                 # as v5b64, where every weight ran at the 3e-5 peak (see the top)
DROPOUT = 0.0                                      # post-mortem fix 2 (v5, v5b64: 0.1)
PRECISION = "no"      # full precision, as v5b64 (bf16 saved no time there: picture loading set the pace)
WORKERS = max(2, min(12, CORES))                   # every step decodes 192 frames (64 x 3 cameras) on the CPU
os.environ["HF_LEROBOT_HOME"] = f"{WORK}/lerobot_home"   # the dataset's local copy (cell 4)

# use_policy_training_preset is left at its default (true): AdamW(lr, weight_decay) from the policy, no scheduler
COMMON = [
    f"--dataset.repo_id={DATASET}", "--dataset.image_transforms.enable=true", "--dataset.video_backend=torchcodec",
    "--policy.type=act", "--policy.device=cuda", f"--policy.dropout={DROPOUT}",
    f"--policy.optimizer_lr={LR}", f"--policy.optimizer_lr_backbone={LR_BACKBONE}",
    "--policy.optimizer_weight_decay=1e-4",
    f"--batch_size={BATCH}", f"--num_workers={WORKERS}", "--wandb.enable=false",
]
print("workers", WORKERS)


def run_live(cmd, log_path):
    """Runs lerobot-train and shows its log here as it comes, also into log_path. Stopping the cell stops training."""
    env = dict(os.environ, TQDM_MININTERVAL="30", PYTHONUNBUFFERED="1")
    lines = []
    pr = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    try:
        with open(log_path, "a") as f:
            for line in pr.stdout:
                if "INFO" in line and not line.startswith("INFO"):
                    line = line.replace("INFO", "\nINFO", 1)
                print(line, end="", flush=True); f.write(line); f.flush(); lines.append(line)
        pr.wait()
    except BaseException:
        pr.terminate()
        try:
            pr.wait(timeout=60)
        except subprocess.TimeoutExpired:
            pr.kill()
        print("\n-- stopped; the training process is ended too")
        raise
    if pr.returncode != 0:
        raise RuntimeError(f"lerobot-train stopped with an error (exit {pr.returncode}); log: {log_path}")
    return lines

In [ ]:
# 4. The dataset onto this machine's local disk, then the PICTURE GATE (post-mortem fix 1, sim/picture_store.py):
#    the recorder wrote a fingerprint of what the encoder stores for frames 0 and 30 of every episode and camera
#    (meta/picture_gate.jsonl). Decoded HERE, the way training reads them, those frames must give the same
#    fingerprints. That checks two things at once: every stored frame depends on its own picture only, and this
#    machine decodes the videos to exactly the pixels the laptop's tests will feed the policy.
#    If it fails: stop, don't train, tell K7.
import hashlib, json, time
import numpy as np
from huggingface_hub import snapshot_download
t = time.time()
ROOT = f"{os.environ['HF_LEROBOT_HOME']}/{DATASET}"
snapshot_download(DATASET, repo_type="dataset", local_dir=ROOT)
print(f"dataset at {ROOT} in {time.time() - t:.0f} s; local disk free {shutil.disk_usage('/content').free / 1e9:.0f} GB")
info = json.load(open(f"{ROOT}/meta/info.json"))
cams = {k: v["info"] for k, v in info["features"].items() if v["dtype"] == "video"}
print(info["total_episodes"], "episodes,", info["total_frames"], "frames, cameras:", [k.split(".")[-1] for k in cams])
for k, i in cams.items():
    assert (i.get("video.codec") == "h264" and i.get("video.g") == 1 and i.get("video.crf") is None
            and "qp" in (i.get("video.extra_options") or {})), f"{k} is not stored one frame at a time: {i}"
rows = [json.loads(l) for l in open(f"{ROOT}/meta/picture_gate.jsonl") if l.strip()]
assert len(rows) == info["total_episodes"], f"gate file has {len(rows)} episodes, the dataset {info['total_episodes']}"
from lerobot.datasets import LeRobotDataset
ds = LeRobotDataset(DATASET, root=ROOT, video_backend="torchcodec")
GATE_EVERY = 10                                    # every 10th episode: ~140 episodes x 2 frames x 3 cameras
checked, bad = 0, []
for r in rows[::GATE_EVERY]:
    e = ds.meta.episodes[r["episode"]]
    i0, n = int(e["dataset_from_index"]), int(e["dataset_to_index"]) - int(e["dataset_from_index"])
    for k, prints in r["frames"].items():
        if int(k) >= n:
            continue
        item = ds[i0 + int(k)]
        for c, want in prints.items():
            img = (item[f"observation.images.{c}"].permute(1, 2, 0).numpy() * 255.0).round().astype(np.uint8)
            checked += 1
            if hashlib.sha1(np.ascontiguousarray(img).tobytes()).hexdigest() != want:
                bad.append((r["episode"], int(k), c))
print(f"picture gate: {checked} stored frames decoded here, {len(bad)} differ from the recorder's fingerprints",
      bad[:10] if bad else "-> PASS")
assert checked and not bad, "the picture gate FAILED: don't train on this dataset, tell K7"


In [ ]:
# 5. Google Drive, where every checkpoint is copied (cell 7). Colab asks once for permission.
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Drive folder:", DRIVE_DIR, "| Drive free:", f"{shutil.disk_usage('/content/drive').free / 1e9:.0f} GB",
      "(needs about 5 GB: 20 x 0.2 GB weights + 2 x 0.6 GB full state)")

In [ ]:
# 6. Smoke: 100 steps, full precision. The loss must be a number, the learning rate must not move, the pictures
#    must load fast enough, and the checkpoint must pass GATE B: the network answers the same when it runs as in
#    training mode (v5 did not: its dropout stretched the running answer). Nothing goes to the Hub or Drive here.
import re, threading

GATE_B = r'''
# Gate b (post-mortem section 8): does the network answer the same when it RUNS as in TRAINING mode? Only the
# dropout switches change: the model stays in inference mode (ACT's latent stays zero), the dropout layers and the
# attention layers' own dropout are put in training mode (as sim/scripts/postmortem_v5/offline_train_vs_inference.py).
# With dropout 0 nothing may change. Random inputs of the right shapes: the check is on the switches, not the data.
import sys
import torch
from lerobot.policies.act.modeling_act import ACTPolicy
dev = "cuda" if torch.cuda.is_available() else "cpu"
p = ACTPolicy.from_pretrained(sys.argv[1]).to(dev).eval()
cfg = p.config
drop = [m for m in p.modules() if isinstance(m, torch.nn.Dropout) and m.p > 0]
mha = [m for m in p.modules() if isinstance(m, torch.nn.MultiheadAttention) and m.dropout > 0]
torch.manual_seed(0)
batch = {k: torch.rand(2, *ft.shape, device=dev) for k, ft in cfg.input_features.items()}
batch["observation.images"] = [batch[k] for k in cfg.image_features]


def answer(training_dropout):
    p.eval()
    if training_dropout:
        for m in p.model.modules():
            if isinstance(m, (torch.nn.Dropout, torch.nn.MultiheadAttention)):
                m.train()
    with torch.no_grad():
        return p.model(batch)[0]


a = answer(False)
gap = max((a - answer(True)).abs().max().item() for _ in range(4))
p.eval()
print(f"dropout in the config {cfg.dropout}; dropout layers switched on {len(drop)}; attention layers with dropout "
      f"{len(mha)}; largest gap between the running and the training-mode answer {gap:.2e} (answers span "
      f"{a.min().item():.2f} .. {a.max().item():.2f})")
ok = cfg.dropout == 0 and not drop and not mha and gap < 1e-4
print("gate b:", "PASS" if ok else "FAIL")
sys.exit(0 if ok else 1)
'''


def smoke(name, steps):
    shutil.rmtree(f"{WORK}/{name}", ignore_errors=True)
    cmd = [TRAIN, *COMMON, f"--steps={steps}", f"--save_freq={steps}", "--log_freq=25",
           "--policy.push_to_hub=false", f"--output_dir={WORK}/{name}", f"--job_name={name}"]
    peak, stop = [0], threading.Event()

    def watch_gpu():                              # the highest GPU memory in use while the smoke runs
        while not stop.wait(2):
            r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                               capture_output=True, text=True)
            peak[0] = max(peak[0], int(r.stdout.split()[0])) if r.stdout.strip() else peak[0]
    threading.Thread(target=watch_gpu, daemon=True).start()
    try:
        log = [l for l in run_live(cmd, f"{WORK}/{name}.log") if "updt_s:" in l]
    finally:
        stop.set()
    loss, updt, data = (re.search(p, log[-1]).group(1) for p in (r"loss:([\d.na]+)", r"updt_s:([\d.]+)", r"data_s:([\d.]+)"))
    lrs = sorted({re.search(r"lr:([\d.e+-]+)", l).group(1) for l in log})
    return loss, float(updt), float(data), lrs, peak[0] / 1024

loss, updt, data, lrs, gpu_gb = smoke("smoke_v6", 100)
print(f"loss {loss}, {updt:.3f} s per step on the GPU + {data:.3f} s loading -> {STEPS} steps = "
      f"{STEPS * (updt + data) / 3600:.1f} h, a checkpoint every {SAVE_FREQ * (updt + data) / 60:.0f} min; "
      f"peak GPU memory {gpu_gb:.1f} GB; learning rate logged: {lrs}")
assert loss != "nan", "loss is nan: tell K7"
assert len(lrs) == 1 and abs(float(lrs[0]) - LR) < 1e-12, f"the learning rate is not a constant {LR}: {lrs}"
if data > 0.5 * updt:
    print("!! loading the pictures takes more than half the GPU time: the CPU is the limit")
open(f"{WORK}/gate_b.py", "w").write(GATE_B)
r = subprocess.run([sys.executable, f"{WORK}/gate_b.py", f"{WORK}/smoke_v6/checkpoints/last/pretrained_model"],
                   capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:] if r.returncode else "")
assert r.returncode == 0, "gate b FAILED: the running answer differs from the training-mode answer. Don't train."
shutil.rmtree(f"{WORK}/smoke_v6", ignore_errors=True)


In [ ]:
# 7. The run. Fresh if MODEL_REPO holds no checkpoint, else RESUMED from its newest one (run this cell again after a
#    disconnect). lerobot pushes each checkpoint to the Hub before it trains on; a thread here copies it to Drive
#    (weights of every step, the full state of 50k and of the newest) and keeps only the newest two on the local disk.
import datetime as dt, pathlib, threading, time
from huggingface_hub import HfApi

try:
    have = [f for f in HfApi().list_repo_files(MODEL_REPO) if f.startswith("checkpoints/")]
except Exception:
    have = []
stamp = dt.datetime.now().strftime("%m%d_%H%M%S")
OUT = f"{WORK}/train/{RUN}_{stamp}"
if have:
    print("resuming from", MODEL_REPO, "checkpoint", max(f.split("/")[1] for f in have))
    cmd = [TRAIN, "--resume=true", f"--config_path={MODEL_REPO}", f"--output_dir={OUT}"]
else:
    print("fresh run ->", MODEL_REPO)
    cmd = [TRAIN, *COMMON, f"--steps={STEPS}", f"--save_freq={SAVE_FREQ}", "--log_freq=100",
           "--save_checkpoint_to_hub=true", "--policy.push_to_hub=true", f"--policy.repo_id={MODEL_REPO}",
           "--policy.private=true", f"--output_dir={OUT}", f"--job_name={RUN}"]
    if PRECISION != "no":                         # lerobot reads a literal "no" as False: pass the flag only for bf16
        cmd.append(f"--accelerator.mixed_precision={PRECISION}")


def to_drive(out):
    """Every minute: copy finished checkpoints (lerobot points `last` at one once it is complete and on the Hub)."""
    done_drive = set()
    while True:
        time.sleep(60)
        try:
            ck = pathlib.Path(out, "checkpoints")
            last = ck / "last"
            if not last.is_symlink():
                continue
            newest = pathlib.Path(os.readlink(last)).name
            steps = sorted(p.name for p in ck.glob("[0-9]*") if p.is_dir() and p.name <= newest)
            for s in steps:
                if s in done_drive:
                    continue
                dst = pathlib.Path(DRIVE_DIR, "checkpoints", s)
                shutil.copytree(ck / s / "pretrained_model", dst / "pretrained_model", dirs_exist_ok=True)
                if s == newest or s in KEEP_STATE:
                    shutil.copytree(ck / s / "training_state", dst / "training_state", dirs_exist_ok=True)
                done_drive.add(s)
                print(f"\n[drive] checkpoint {s} copied to {dst}", flush=True)
            for p in pathlib.Path(DRIVE_DIR, "checkpoints").glob("[0-9]*/training_state"):
                if p.parent.name not in KEEP_STATE and p.parent.name != newest:
                    shutil.rmtree(p, ignore_errors=True)       # older full states: the weights stay
            for s in steps[:-2]:
                if s in done_drive:
                    shutil.rmtree(ck / s, ignore_errors=True)  # on the Hub and on Drive by now
        except Exception as e:                                  # Drive hiccup: try again next minute
            print(f"\n[drive] copy failed, retrying: {e}", flush=True)

threading.Thread(target=to_drive, args=(OUT,), daemon=True).start()
run_live(cmd, f"{WORK}/train_{RUN}_{stamp}.log")

In [ ]:
# 8. What is on the Hub and on Drive now.
from huggingface_hub import HfApi
print("Hub:", sorted({f.split("/")[1] for f in HfApi().list_repo_files(MODEL_REPO) if f.startswith("checkpoints/")}))
print("Drive:", sorted(os.listdir(f"{DRIVE_DIR}/checkpoints")) if os.path.isdir(f"{DRIVE_DIR}/checkpoints") else [])

In [ ]:
import glob; print([shutil.copy(f, DRIVE_DIR) for f in glob.glob('/content/train_*.log') + glob.glob('/content/smoke_*.log')])